# Lab 3 — Trees, Random Forests and Gradient Boosting
**Machine Learning I · PEU-CD 2026 · ENEI**

Companion to `tutorial.pdf`.

In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.inspection import permutation_importance
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(155)

Xa, ya = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xa, ya, test_size=0.2, random_state=155, stratify=ya)
N = len(ytr); print("train", Xtr.shape, "test", Xte.shape)

## 1. Impurity and information gain
### Task 1

In [ ]:
def props(counts):
    counts = np.asarray(counts, float); return counts / counts.sum()
# TODO: entropy(counts), gini(counts), misclass(counts) on a class-count vector; gain(parent, children, impurity)
def entropy(counts): ...
def gini(counts): ...
def misclass(counts): ...
def gain(parent, children, impurity): ...

assert np.isclose(gain([8, 4], [[6, 1], [2, 3]], entropy), 0.1685, atol=1e-4)
assert np.isclose(gain([8, 4], [[6, 1], [2, 3]], gini), 0.1016, atol=1e-4)
assert np.isclose(gain([8, 4], [[6, 1], [2, 3]], misclass), 0.0833, atol=1e-4)
# TODO: the (400,400) example: impurity after split A vs split B under the three criteria


### Task 2 — the root split, by hand

In [ ]:
def best_split(X, y, impurity):
    # TODO: scan every feature and every midpoint between consecutive sorted values; return (j, s, gain)
    ...

j, s, g = best_split(Xtr, ytr, gini)
stump = DecisionTreeClassifier(max_depth=1).fit(Xtr, ytr)
print(f"mine   : feature {j}, threshold {s:.4f}, gain {g:.4f}")
print(f"sklearn: feature {stump.tree_.feature[0]}, threshold {stump.tree_.threshold[0]:.4f}")
assert j == stump.tree_.feature[0] and np.isclose(s, stump.tree_.threshold[0], atol=1e-3)

## 2. One tree
### Task 3 — overfitting, in two directions

In [ ]:
# TODO: train/test error vs max_depth in 1..12, and vs min_samples_leaf in [1,2,5,10,20,50]; two plots
# TODO: the unrestricted tree: leaves, train error, test error -> store test error as single_tree_err
full = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
single_tree_err = 1 - full.score(Xte, yte)


## 3. Bagging and the variance formula
### Task 4 — bagging from scratch

In [ ]:
def bagged_trees(X, y, B, seed=155):
    # TODO: B bootstrap samples (rng.integers(0, N, N)), one unrestricted tree each; return trees and index arrays
    ...
def vote(trees, X):
    # TODO: majority vote
    ...
Bs = [1, 2, 5, 10, 25, 50, 100, 200]
trees200, idxs200 = bagged_trees(Xtr, ytr, 200)
# TODO: test error vs B, overlay single_tree_err
# TODO: out-of-bag error by hand; compare with BaggingClassifier(oob_score=True) and with the test error
# TODO: mean fraction of distinct training points per bootstrap sample (expect ~0.632)


### Task 5 — measure rho and test the variance formula

In [ ]:
def rho_and_var(trees, X):
    # TODO: predictions as +-1 (B x n_test); average pairwise correlation across trees; mean per-tree variance;
    # variance of the averaged vote. Return (rho, sigma2, var_avg).
    ...
rho, s2, v = rho_and_var(trees200[:100], Xte)
print(f"rho={rho:.3f}  Var(average)={v:.4f}  formula={rho * s2 + (1 - rho) * s2 / 100:.4f}")

### Task 6 — random forests lower rho

In [ ]:
# TODO: for m in [1,3,5,10,30]: RandomForestClassifier(max_features=m); rho via rho_and_var(rf.estimators_, Xte),
# mean single-tree test error, forest test error. Then permutation_importance on the best forest: top-5 features.


## 4. Boosting
### Task 7 — AdaBoost by hand (must match Lecture 5's table)

In [ ]:
x = np.array([1, 2, 3, 4, 5.0]); y = np.array([1, 1, -1, -1, 1])
thresholds = [1.5, 2.5, 3.5, 4.5]
stumps = [(s, t) for t in thresholds for s in (+1, -1)]            # h(x) = s * sign(x - t)
H = lambda s, t, x: s * np.where(x > t, 1, -1)
# TODO: three rounds of AdaBoost: weighted error of each stump, pick the best, alpha, weight update with Z.
# Print eps, alpha, Z, new weights each round. Then check Z = 2 sqrt(eps(1-eps)) and mean exp-loss = prod Z.
w = np.ones(5) / 5; f = np.zeros(5)


### Task 8 — early stopping on M

In [ ]:
Xa_, Xv_, ya_, yv_ = train_test_split(Xtr, ytr, test_size=0.2, random_state=155, stratify=ytr)
# TODO: for nu in [1, 0.1, 0.01]: GradientBoostingClassifier(n_estimators=2000, learning_rate=nu, max_depth=3);
# validation log loss after every round via staged_predict_proba; best M; refit on Xtr with that M; test error.


### Task 9 — three ensembles, one table

In [ ]:
# TODO: GridSearchCV (cv=5) for a tree, bagging, random forest and gradient boosting over small grids;
# table of test accuracy, AUC and wall-clock fit time; two sentences on which to deploy.


## 5. Exercises — see `tutorial.pdf`, Section 5